# Network Performance Anomaly Detection Analysis

## Project: Daily Signal - Anomaly Detection on Daily Network Performance Data

### Overview
This notebook performs anomaly detection on daily network performance measurements collected across Egyptian regions over 21 days (March 1-21, 2026). The data covers multiple carriers and three technology types: LTE, 5G, and Multi-RAT.

### Methods Used
- **Isolation Forest**: Tree-based anomaly detection that isolates anomalies by randomly selecting features and split values
- **One-Class SVM (OC-SVM)**: Kernel-based method that learns a decision boundary around normal data
- **Autoencoder**: Neural network that learns to reconstruct normal data; anomalies have high reconstruction error

### Questions to Answer
1. Which regions show the most variable download speed across the 21 days?
2. Are there specific days where download speed or latency looks abnormal across multiple regions simultaneously?
3. Do the carriers perform consistently across regions?
4. Is there a difference in anomaly patterns between LTE and 5G?
5. What are the top 5 'worst performing' region-carrier-day combinations?

---
## 1. Data Loading and Exploration

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import warnings
warnings.filterwarnings('ignore')

# Set plot style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

# For reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print("Libraries imported successfully!")

In [ ]:
# Load the dataset
df = pd.read_csv('/home/z/my-project/upload/Performance.csv')

print("Dataset Shape:", df.shape)
print("\nColumn Names:")
print(df.columns.tolist())
print("\nFirst few rows:")
df.head()

In [ ]:
# Dataset info
print("Dataset Information:")
print("="*50)
df.info()

In [ ]:
# Statistical summary
print("Statistical Summary of Numerical Columns:")
print("="*50)
df.describe()

In [ ]:
# Check for missing values
print("Missing Values:")
print("="*50)
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "No missing values found!")

# Check unique values for categorical columns
print("\nUnique Values in Categorical Columns:")
print("="*50)
categorical_cols = ['aggregate_date', 'place_name', 'region', 'place_type', 
                    'carrier_name', 'aggregation_period', 'technology_type']
for col in categorical_cols:
    print(f"{col}: {df[col].nunique()} unique values")

In [ ]:
# Explore key categorical variables
print("Unique Regions:")
print(df['region'].unique())
print(f"\nTotal Regions: {df['region'].nunique()}")

print("\n" + "="*50)
print("Unique Carriers:")
print(df['carrier_name'].unique())

print("\n" + "="*50)
print("Unique Technology Types:")
print(df['technology_type'].unique())

print("\n" + "="*50)
print("Unique Place Types:")
print(df['place_type'].unique())

print("\n" + "="*50)
print("Date Range:")
print(f"From: {df['aggregate_date'].min()}")
print(f"To: {df['aggregate_date'].max()}")
print(f"Number of unique dates: {df['aggregate_date'].nunique()}")

In [ ]:
# Sample count distribution
print("Sample Count Distribution:")
print("="*50)
print(df['sample_count'].describe())

# Visualize sample count distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df['sample_count'], bins=50, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Sample Count')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Sample Count')

# Box plot
axes[1].boxplot(df['sample_count'], vert=True)
axes[1].set_ylabel('Sample Count')
axes[1].set_title('Box Plot of Sample Count')

plt.tight_layout()
plt.show()

# Show rows with very low sample counts
low_sample_rows = df[df['sample_count'] <= 5]
print(f"\nRows with sample_count <= 5: {len(low_sample_rows)} ({len(low_sample_rows)/len(df)*100:.2f}%)")
print(f"Rows with sample_count = 1: {len(df[df['sample_count'] == 1])} ({len(df[df['sample_count'] == 1])/len(df)*100:.2f}%)")

---
## 2. Data Filtering and Preprocessing

### Filtering Decisions and Justifications

Based on the data exploration, we need to make informed decisions about filtering:

1. **Sample Count Threshold**: Rows with very low sample_count are unreliable as they represent few measurements. We'll set a minimum threshold.

2. **Place Type**: We'll focus on 'locality' level data (city-level) and exclude 'country' aggregates to focus on regional patterns.

3. **Aggregation Period**: We'll focus on 'Day' level data as our analysis is about daily patterns.

In [ ]:
# ============================================
# FILTERING DECISION 1: Sample Count Threshold
# ============================================

# Analysis to determine appropriate threshold
print("Sample Count Analysis for Threshold Decision:")
print("="*60)

# Calculate percentiles
percentiles = [1, 5, 10, 25, 50, 75, 90, 95, 99]
for p in percentiles:
    print(f"{p}th percentile: {df['sample_count'].quantile(p/100)}")

# Check data retention at different thresholds
print("\nData Retention at Different Thresholds:")
print("="*60)
thresholds = [1, 3, 5, 10, 15, 20]
for thresh in thresholds:
    retained = len(df[df['sample_count'] >= thresh])
    pct = retained / len(df) * 100
    print(f"sample_count >= {thresh}: {retained} rows ({pct:.1f}%)")

# Decision: Set minimum sample_count threshold to 5
# Justification: 
# - sample_count=1 represents a single speed test - statistically unreliable
# - A threshold of 5 retains ~88% of data while removing the least reliable measurements
# - This balances data quality with sample size for meaningful analysis

SAMPLE_COUNT_THRESHOLD = 5
print(f"\n>>> SELECTED THRESHOLD: sample_count >= {SAMPLE_COUNT_THRESHOLD}")
print(">>> Justification: Removes single measurements (statistically unreliable) while retaining ~88% of data")

In [ ]:
# ============================================
# FILTERING DECISION 2: Place Type
# ============================================

print("Place Type Distribution:")
print("="*60)
print(df['place_type'].value_counts())

# Decision: Filter to 'locality' only
# Justification: 
# - 'locality' represents city-level measurements
# - 'country' represents Egypt-level aggregates which would double-count data
# - We want to analyze regional variations, so locality-level data is appropriate

print("\n>>> SELECTED FILTER: place_type == 'locality'")
print(">>> Justification: Focus on city-level measurements for regional analysis; exclude country aggregates")

In [ ]:
# Apply all filters
print("Applying Filters...")
print("="*60)

# Store original size
original_size = len(df)

# Apply filters
df_filtered = df[
    (df['sample_count'] >= SAMPLE_COUNT_THRESHOLD) &
    (df['place_type'] == 'locality')
].copy()

# Report results
print(f"Original dataset size: {original_size:,} rows")
print(f"Filtered dataset size: {len(df_filtered):,} rows")
print(f"Rows removed: {original_size - len(df_filtered):,} ({(original_size - len(df_filtered))/original_size*100:.1f}%)")

# Check remaining data
print("\nFiltered Data Summary:")
print("="*60)
print(f"Unique dates: {df_filtered['aggregate_date'].nunique()}")
print(f"Unique regions: {df_filtered['region'].nunique()}")
print(f"Unique carriers: {df_filtered['carrier_name'].nunique()}")
print(f"Unique technology types: {df_filtered['technology_type'].nunique()}")

In [ ]:
# Convert date column to datetime
df_filtered['aggregate_date'] = pd.to_datetime(df_filtered['aggregate_date'])

# Create additional time-based features
df_filtered['day_of_week'] = df_filtered['aggregate_date'].dt.dayofweek
df_filtered['day_num'] = df_filtered['aggregate_date'].dt.day

print("Date column converted and features added.")
df_filtered.head()

---
## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Distribution of key KPIs
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

kpis = ['mean_download_kbps', 'mean_upload_kbps', 'mean_latency_ms',
        'median_download_kbps', 'median_upload_kbps', 'median_latency_ms']

for idx, kpi in enumerate(kpis):
    row = idx // 3
    col = idx % 3
    axes[row, col].hist(df_filtered[kpi], bins=50, edgecolor='black', alpha=0.7)
    axes[row, col].set_xlabel(kpi)
    axes[row, col].set_ylabel('Frequency')
    axes[row, col].set_title(f'Distribution of {kpi}')

plt.tight_layout()
plt.savefig('/home/z/my-project/download/anomaly_detection_project/output/kpi_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Performance by Region
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Download speed by region
region_download = df_filtered.groupby('region')['mean_download_kbps'].mean().sort_values(ascending=False)
axes[0].barh(region_download.index, region_download.values)
axes[0].set_xlabel('Mean Download Speed (kbps)')
axes[0].set_title('Average Download Speed by Region')

# Upload speed by region
region_upload = df_filtered.groupby('region')['mean_upload_kbps'].mean().sort_values(ascending=False)
axes[1].barh(region_upload.index, region_upload.values)
axes[1].set_xlabel('Mean Upload Speed (kbps)')
axes[1].set_title('Average Upload Speed by Region')

# Latency by region
region_latency = df_filtered.groupby('region')['mean_latency_ms'].mean().sort_values(ascending=True)
axes[2].barh(region_latency.index, region_latency.values)
axes[2].set_xlabel('Mean Latency (ms)')
axes[2].set_title('Average Latency by Region')

plt.tight_layout()
plt.savefig('/home/z/my-project/download/anomaly_detection_project/output/performance_by_region.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Performance by Carrier
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Download speed by carrier
carrier_download = df_filtered.groupby('carrier_name')['mean_download_kbps'].agg(['mean', 'std'])
x_pos = np.arange(len(carrier_download))
axes[0].bar(x_pos, carrier_download['mean'], yerr=carrier_download['std'], capsize=5, alpha=0.7)
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(carrier_download.index)
axes[0].set_ylabel('Mean Download Speed (kbps)')
axes[0].set_title('Download Speed by Carrier (with std)')

# Upload speed by carrier
carrier_upload = df_filtered.groupby('carrier_name')['mean_upload_kbps'].agg(['mean', 'std'])
axes[1].bar(x_pos, carrier_upload['mean'], yerr=carrier_upload['std'], capsize=5, alpha=0.7, color='orange')
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(carrier_upload.index)
axes[1].set_ylabel('Mean Upload Speed (kbps)')
axes[1].set_title('Upload Speed by Carrier (with std)')

# Latency by carrier
carrier_latency = df_filtered.groupby('carrier_name')['mean_latency_ms'].agg(['mean', 'std'])
axes[2].bar(x_pos, carrier_latency['mean'], yerr=carrier_latency['std'], capsize=5, alpha=0.7, color='green')
axes[2].set_xticks(x_pos)
axes[2].set_xticklabels(carrier_latency.index)
axes[2].set_ylabel('Mean Latency (ms)')
axes[2].set_title('Latency by Carrier (with std)')

plt.tight_layout()
plt.savefig('/home/z/my-project/download/anomaly_detection_project/output/performance_by_carrier.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Performance by Technology Type
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Download speed by technology
tech_download = df_filtered.groupby('technology_type')['mean_download_kbps'].agg(['mean', 'std'])
x_pos = np.arange(len(tech_download))
axes[0].bar(x_pos, tech_download['mean'], yerr=tech_download['std'], capsize=5, alpha=0.7)
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(tech_download.index)
axes[0].set_ylabel('Mean Download Speed (kbps)')
axes[0].set_title('Download Speed by Technology')

# Upload speed by technology
tech_upload = df_filtered.groupby('technology_type')['mean_upload_kbps'].agg(['mean', 'std'])
axes[1].bar(x_pos, tech_upload['mean'], yerr=tech_upload['std'], capsize=5, alpha=0.7, color='orange')
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(tech_upload.index)
axes[1].set_ylabel('Mean Upload Speed (kbps)')
axes[1].set_title('Upload Speed by Technology')

# Latency by technology
tech_latency = df_filtered.groupby('technology_type')['mean_latency_ms'].agg(['mean', 'std'])
axes[2].bar(x_pos, tech_latency['mean'], yerr=tech_latency['std'], capsize=5, alpha=0.7, color='green')
axes[2].set_xticks(x_pos)
axes[2].set_xticklabels(tech_latency.index)
axes[2].set_ylabel('Mean Latency (ms)')
axes[2].set_title('Latency by Technology')

plt.tight_layout()
plt.savefig('/home/z/my-project/download/anomaly_detection_project/output/performance_by_technology.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Daily trends across the 21 days
daily_trends = df_filtered.groupby('aggregate_date').agg({
    'mean_download_kbps': 'mean',
    'mean_upload_kbps': 'mean',
    'mean_latency_ms': 'mean'
}).reset_index()

fig, axes = plt.subplots(3, 1, figsize=(14, 12))

axes[0].plot(daily_trends['aggregate_date'], daily_trends['mean_download_kbps'], marker='o', linewidth=2)
axes[0].set_ylabel('Mean Download Speed (kbps)')
axes[0].set_title('Daily Average Download Speed Trend')
axes[0].tick_params(axis='x', rotation=45)

axes[1].plot(daily_trends['aggregate_date'], daily_trends['mean_upload_kbps'], marker='o', linewidth=2, color='orange')
axes[1].set_ylabel('Mean Upload Speed (kbps)')
axes[1].set_title('Daily Average Upload Speed Trend')
axes[1].tick_params(axis='x', rotation=45)

axes[2].plot(daily_trends['aggregate_date'], daily_trends['mean_latency_ms'], marker='o', linewidth=2, color='green')
axes[2].set_ylabel('Mean Latency (ms)')
axes[2].set_xlabel('Date')
axes[2].set_title('Daily Average Latency Trend')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('/home/z/my-project/download/anomaly_detection_project/output/daily_trends.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation matrix
numeric_cols = ['sample_count', 'mean_download_kbps', 'mean_upload_kbps', 'mean_latency_ms',
                'median_download_kbps', 'median_upload_kbps', 'median_latency_ms']

corr_matrix = df_filtered[numeric_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Correlation Matrix of KPIs')
plt.tight_layout()
plt.savefig('/home/z/my-project/download/anomaly_detection_project/output/correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Anomaly Detection Methods

We will implement three anomaly detection methods:
1. **Isolation Forest** - Tree-based method that isolates anomalies
2. **One-Class SVM (OC-SVM)** - Learns a boundary around normal data
3. **Autoencoder** - Neural network reconstruction error approach

In [ ]:
# Prepare features for anomaly detection
# We'll focus on the key KPIs: download speed, upload speed, latency

feature_cols = ['mean_download_kbps', 'mean_upload_kbps', 'mean_latency_ms',
                'median_download_kbps', 'median_upload_kbps', 'median_latency_ms']

# Scale the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_filtered[feature_cols])

print(f"Feature matrix shape: {X_scaled.shape}")
print(f"Features used: {feature_cols}")

### 4.1 Isolation Forest

**Why Isolation Forest?**
- Works well with high-dimensional data
- Does not require assumption of data distribution
- Efficient for large datasets
- Effective at detecting point anomalies

**Parameters:**
- `n_estimators`: Number of trees (100 is reasonable)
- `contamination`: Expected proportion of anomalies (we'll use 0.05 - 5%)
- `random_state`: For reproducibility

In [ ]:
# Isolation Forest
print("Training Isolation Forest...")
print("="*60)

iso_forest = IsolationForest(
    n_estimators=100,
    contamination=0.05,  # Expect 5% anomalies
    random_state=42,
    n_jobs=-1
)

# Fit and predict
iso_forest.fit(X_scaled)
iso_predictions = iso_forest.predict(X_scaled)
iso_scores = iso_forest.decision_function(X_scaled)

# Convert predictions: -1 = anomaly, 1 = normal
df_filtered['iso_forest_anomaly'] = (iso_predictions == -1).astype(int)
df_filtered['iso_forest_score'] = iso_scores

# Results
n_anomalies_iso = df_filtered['iso_forest_anomaly'].sum()
print(f"Isolation Forest Results:")
print(f"Total samples: {len(df_filtered)}")
print(f"Anomalies detected: {n_anomalies_iso} ({n_anomalies_iso/len(df_filtered)*100:.2f}%)")
print(f"Normal samples: {len(df_filtered) - n_anomalies_iso}")

In [ ]:
# Visualize Isolation Forest results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Score distribution
axes[0].hist(df_filtered['iso_forest_score'], bins=50, edgecolor='black', alpha=0.7)
axes[0].axvline(x=0, color='red', linestyle='--', label='Threshold')
axes[0].set_xlabel('Anomaly Score')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Isolation Forest: Anomaly Score Distribution')
axes[0].legend()

# PCA visualization
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

scatter = axes[1].scatter(X_pca[:, 0], X_pca[:, 1], 
                          c=df_filtered['iso_forest_anomaly'], 
                          cmap='coolwarm', alpha=0.6, s=20)
axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
axes[1].set_title('Isolation Forest: PCA Visualization')
plt.colorbar(scatter, ax=axes[1], label='Anomaly (1=Yes)')

plt.tight_layout()
plt.savefig('/home/z/my-project/download/anomaly_detection_project/output/isolation_forest_results.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.2 One-Class SVM (OC-SVM)

**Why OC-SVM?**
- Effective for learning the shape of normal data
- Works well when normal data forms a compact cluster
- Kernel trick allows non-linear decision boundaries

**Parameters:**
- `kernel`: RBF for non-linear boundaries
- `nu`: Upper bound on fraction of margin errors (similar to contamination)
- `gamma`: Kernel coefficient

In [ ]:
# One-Class SVM
print("Training One-Class SVM...")
print("="*60)

ocsvm = OneClassSVM(
    kernel='rbf',
    nu=0.05,  # Similar to contamination
    gamma='scale'
)

# Fit and predict
ocsvm.fit(X_scaled)
ocsvm_predictions = ocsvm.predict(X_scaled)
ocsvm_scores = ocsvm.decision_function(X_scaled)

# Convert predictions: -1 = anomaly, 1 = normal
df_filtered['ocsvm_anomaly'] = (ocsvm_predictions == -1).astype(int)
df_filtered['ocsvm_score'] = ocsvm_scores

# Results
n_anomalies_ocsvm = df_filtered['ocsvm_anomaly'].sum()
print(f"One-Class SVM Results:")
print(f"Total samples: {len(df_filtered)}")
print(f"Anomalies detected: {n_anomalies_ocsvm} ({n_anomalies_ocsvm/len(df_filtered)*100:.2f}%)")
print(f"Normal samples: {len(df_filtered) - n_anomalies_ocsvm}")

In [ ]:
# Visualize OC-SVM results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Score distribution
axes[0].hist(df_filtered['ocsvm_score'], bins=50, edgecolor='black', alpha=0.7, color='orange')
axes[0].axvline(x=0, color='red', linestyle='--', label='Threshold')
axes[0].set_xlabel('Decision Function Score')
axes[0].set_ylabel('Frequency')
axes[0].set_title('One-Class SVM: Score Distribution')
axes[0].legend()

# PCA visualization
scatter = axes[1].scatter(X_pca[:, 0], X_pca[:, 1], 
                          c=df_filtered['ocsvm_anomaly'], 
                          cmap='coolwarm', alpha=0.6, s=20)
axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
axes[1].set_title('One-Class SVM: PCA Visualization')
plt.colorbar(scatter, ax=axes[1], label='Anomaly (1=Yes)')

plt.tight_layout()
plt.savefig('/home/z/my-project/download/anomaly_detection_project/output/ocsvm_results.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.3 Autoencoder

**Why Autoencoder?**
- Learns to reconstruct normal data patterns
- Anomalies have high reconstruction error
- Can capture complex non-linear relationships
- Flexible architecture

**Architecture:**
- Encoder: 6 -> 4 -> 2 neurons
- Decoder: 2 -> 4 -> 6 neurons
- Reconstruction error threshold based on percentile

In [ ]:
# Autoencoder for Anomaly Detection
print("Building and Training Autoencoder...")
print("="*60)

# Normalize data for neural network
minmax_scaler = MinMaxScaler()
X_normalized = minmax_scaler.fit_transform(df_filtered[feature_cols])

# Define Autoencoder architecture
input_dim = X_normalized.shape[1]
encoding_dim = 2

# Build the model
input_layer = keras.Input(shape=(input_dim,))
encoder = layers.Dense(4, activation='relu')(input_layer)
encoder = layers.Dense(encoding_dim, activation='relu')(encoder)
decoder = layers.Dense(4, activation='relu')(encoder)
decoder = layers.Dense(input_dim, activation='sigmoid')(decoder)

autoencoder = keras.Model(input_layer, decoder)
autoencoder.compile(optimizer='adam', loss='mse')

print("Autoencoder Architecture:")
autoencoder.summary()

In [ ]:
# Train the autoencoder
history = autoencoder.fit(
    X_normalized, X_normalized,
    epochs=100,
    batch_size=32,
    shuffle=True,
    validation_split=0.1,
    verbose=0
)

print("Training completed!")

# Plot training history
plt.figure(figsize=(10, 4))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.title('Autoencoder Training History')
plt.legend()
plt.tight_layout()
plt.savefig('/home/z/my-project/download/anomaly_detection_project/output/autoencoder_training.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Calculate reconstruction error
reconstructions = autoencoder.predict(X_normalized, verbose=0)
mse = np.mean(np.power(X_normalized - reconstructions, 2), axis=1)

# Set threshold at 95th percentile (5% anomalies)
threshold = np.percentile(mse, 95)

df_filtered['autoencoder_mse'] = mse
df_filtered['autoencoder_anomaly'] = (mse > threshold).astype(int)

# Results
n_anomalies_auto = df_filtered['autoencoder_anomaly'].sum()
print(f"Autoencoder Results:")
print(f"Total samples: {len(df_filtered)}")
print(f"Anomalies detected: {n_anomalies_auto} ({n_anomalies_auto/len(df_filtered)*100:.2f}%)")
print(f"MSE Threshold: {threshold:.6f}")
print(f"Normal samples: {len(df_filtered) - n_anomalies_auto}")

In [ ]:
# Visualize Autoencoder results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# MSE distribution
axes[0].hist(df_filtered['autoencoder_mse'], bins=50, edgecolor='black', alpha=0.7, color='green')
axes[0].axvline(x=threshold, color='red', linestyle='--', label=f'Threshold ({threshold:.4f})')
axes[0].set_xlabel('Reconstruction Error (MSE)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Autoencoder: Reconstruction Error Distribution')
axes[0].legend()

# PCA visualization
scatter = axes[1].scatter(X_pca[:, 0], X_pca[:, 1], 
                          c=df_filtered['autoencoder_anomaly'], 
                          cmap='coolwarm', alpha=0.6, s=20)
axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
axes[1].set_title('Autoencoder: PCA Visualization')
plt.colorbar(scatter, ax=axes[1], label='Anomaly (1=Yes)')

plt.tight_layout()
plt.savefig('/home/z/my-project/download/anomaly_detection_project/output/autoencoder_results.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.4 Method Comparison

In [ ]:
# Compare the three methods
print("Method Comparison Summary:")
print("="*60)

comparison_df = pd.DataFrame({
    'Method': ['Isolation Forest', 'One-Class SVM', 'Autoencoder'],
    'Anomalies Detected': [
        df_filtered['iso_forest_anomaly'].sum(),
        df_filtered['ocsvm_anomaly'].sum(),
        df_filtered['autoencoder_anomaly'].sum()
    ],
    'Percentage': [
        df_filtered['iso_forest_anomaly'].mean() * 100,
        df_filtered['ocsvm_anomaly'].mean() * 100,
        df_filtered['autoencoder_anomaly'].mean() * 100
    ]
})

print(comparison_df.to_string(index=False))

# Find overlapping anomalies (detected by all methods)
df_filtered['all_methods_anomaly'] = (
    (df_filtered['iso_forest_anomaly'] == 1) & 
    (df_filtered['ocsvm_anomaly'] == 1) & 
    (df_filtered['autoencoder_anomaly'] == 1)
).astype(int)

# Find anomalies detected by at least 2 methods
df_filtered['anomaly_count'] = (
    df_filtered['iso_forest_anomaly'] + 
    df_filtered['ocsvm_anomaly'] + 
    df_filtered['autoencoder_anomaly']
)

df_filtered['consensus_anomaly'] = (df_filtered['anomaly_count'] >= 2).astype(int)

print(f"\nAnomalies detected by ALL 3 methods: {df_filtered['all_methods_anomaly'].sum()}")
print(f"Anomalies detected by at least 2 methods: {df_filtered['consensus_anomaly'].sum()}")

In [ ]:
# Visualization of method overlap
from matplotlib_venn import venn3

iso_set = set(df_filtered[df_filtered['iso_forest_anomaly'] == 1].index)
ocsvm_set = set(df_filtered[df_filtered['ocsvm_anomaly'] == 1].index)
auto_set = set(df_filtered[df_filtered['autoencoder_anomaly'] == 1].index)

plt.figure(figsize=(10, 8))
venn3([iso_set, ocsvm_set, auto_set], 
      ('Isolation Forest', 'One-Class SVM', 'Autoencoder'))
plt.title('Overlap of Anomalies Detected by Each Method')
plt.tight_layout()
plt.savefig('/home/z/my-project/download/anomaly_detection_project/output/method_overlap_venn.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Create combined anomaly log
anomaly_log = df_filtered[df_filtered['consensus_anomaly'] == 1].copy()

# Determine which method(s) flagged each anomaly
def get_methods_flagged(row):
    methods = []
    if row['iso_forest_anomaly'] == 1:
        methods.append('Isolation Forest')
    if row['ocsvm_anomaly'] == 1:
        methods.append('OC-SVM')
    if row['autoencoder_anomaly'] == 1:
        methods.append('Autoencoder')
    return ', '.join(methods)

anomaly_log['methods_flagged'] = anomaly_log.apply(get_methods_flagged, axis=1)

print(f"Total anomalies in consensus log: {len(anomaly_log)}")
print("\nSample of Anomaly Log:")
anomaly_log[['aggregate_date', 'region', 'carrier_name', 'technology_type', 
             'mean_download_kbps', 'mean_upload_kbps', 'mean_latency_ms',
             'methods_flagged']].head(10)

---
## 5. Answering the Project Questions

Now we answer each of the 5 questions from the project documentation.

### Q1: Which regions show the most variable download speed across the 21 days?

**Approach:** We calculate the coefficient of variation (CV) for download speed by region. CV = std/mean, which allows us to compare variability across regions with different average speeds.

In [ ]:
# Q1: Most variable download speed regions
print("Q1: Regions with Most Variable Download Speed")
print("="*60)

# Calculate coefficient of variation (CV) for download speed by region
region_variability = df_filtered.groupby('region').agg({
    'mean_download_kbps': ['mean', 'std', 'count']
})

region_variability.columns = ['mean_download', 'std_download', 'count']
region_variability['cv_download'] = region_variability['std_download'] / region_variability['mean_download']
region_variability = region_variability.sort_values('cv_download', ascending=False)

print("\nRegion Download Speed Variability (sorted by CV):")
print(region_variability.round(2).to_string())

# Top 5 most variable regions
top5_variable = region_variability.head(5)
print(f"\n>>> TOP 5 MOST VARIABLE REGIONS:")
for i, (region, row) in enumerate(top5_variable.iterrows(), 1):
    print(f"   {i}. {region}: CV={row['cv_download']:.3f} (Mean={row['mean_download']:.0f} kbps, Std={row['std_download']:.0f} kbps)")

In [ ]:
# Visualization for Q1
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# CV comparison
axes[0].barh(region_variability.index, region_variability['cv_download'].values, color='steelblue')
axes[0].set_xlabel('Coefficient of Variation (CV)')
axes[0].set_ylabel('Region')
axes[0].set_title('Q1: Download Speed Variability by Region (CV)')

# Box plots for top 5 variable regions
top5_regions = top5_variable.index.tolist()
df_top5 = df_filtered[df_filtered['region'].isin(top5_regions)]

df_top5.boxplot(column='mean_download_kbps', by='region', ax=axes[1])
axes[1].set_xlabel('Region')
axes[1].set_ylabel('Mean Download Speed (kbps)')
axes[1].set_title('Q1: Download Speed Distribution (Top 5 Variable Regions)')
plt.suptitle('')  # Remove automatic title

plt.tight_layout()
plt.savefig('/home/z/my-project/download/anomaly_detection_project/output/q1_region_variability.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n>>> Q1 ANSWER:")
print(f"The most variable regions for download speed are: {', '.join(top5_variable.index.tolist()[:3])}.")
print("These regions show the highest coefficient of variation, meaning their download speeds")
print("fluctuate the most relative to their average performance. This could indicate:")
print("- Unstable network infrastructure")
print("- Variable user demand patterns")
print("- Coverage or capacity issues in certain areas")

### Q2: Are there specific days where download speed or latency looks abnormal across multiple regions simultaneously?

**Approach:** We identify days where multiple regions show anomalous behavior, suggesting national-level events rather than local issues.

In [ ]:
# Q2: Days with abnormal cross-region patterns
print("Q2: Days with Abnormal Cross-Region Patterns")
print("="*60)

# Count anomalies by date
daily_anomalies = df_filtered.groupby('aggregate_date').agg({
    'consensus_anomaly': 'sum',
    'region': 'nunique',
    'mean_download_kbps': 'mean',
    'mean_latency_ms': 'mean'
}).reset_index()

daily_anomalies.columns = ['date', 'anomaly_count', 'regions_affected', 'avg_download', 'avg_latency']

# Calculate anomalies per region for each day
anomaly_regions_per_day = df_filtered[df_filtered['consensus_anomaly'] == 1].groupby('aggregate_date')['region'].nunique()
daily_anomalies['unique_regions_with_anomalies'] = daily_anomalies['date'].map(anomaly_regions_per_day).fillna(0).astype(int)

print("\nDaily Anomaly Summary:")
print(daily_anomalies.to_string(index=False))

# Identify days with high cross-region anomalies
threshold_regions = 3  # At least 3 regions affected
problematic_days = daily_anomalies[daily_anomalies['unique_regions_with_anomalies'] >= threshold_regions]

print(f"\n>>> Days affecting {threshold_regions}+ regions simultaneously:")
if len(problematic_days) > 0:
    for _, row in problematic_days.iterrows():
        print(f"   - {row['date'].strftime('%Y-%m-%d')}: {row['unique_regions_with_anomalies']} regions, {row['anomaly_count']} anomalies")
        print(f"     Avg Download: {row['avg_download']:.0f} kbps, Avg Latency: {row['avg_latency']:.1f} ms")
else:
    print("   No days with widespread cross-region anomalies detected.")

In [ ]:
# Visualization for Q2
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Daily anomaly count
axes[0].bar(daily_anomalies['date'], daily_anomalies['anomaly_count'], color='steelblue', alpha=0.7)
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Number of Anomalies')
axes[0].set_title('Q2: Daily Anomaly Count')
axes[0].tick_params(axis='x', rotation=45)

# Highlight problematic days
for _, row in problematic_days.iterrows():
    axes[0].axvline(x=row['date'], color='red', linestyle='--', alpha=0.5)

# Regions affected per day
axes[1].bar(daily_anomalies['date'], daily_anomalies['unique_regions_with_anomalies'], color='coral', alpha=0.7)
axes[1].axhline(y=threshold_regions, color='red', linestyle='--', label=f'Threshold ({threshold_regions} regions)')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Unique Regions with Anomalies')
axes[1].set_title('Q2: Regions Affected by Anomalies Per Day')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend()

plt.tight_layout()
plt.savefig('/home/z/my-project/download/anomaly_detection_project/output/q2_daily_anomalies.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n>>> Q2 ANSWER:")
if len(problematic_days) > 0:
    print(f"Yes, there are {len(problematic_days)} days where anomalies affected multiple regions simultaneously:")
    for _, row in problematic_days.iterrows():
        print(f"  - {row['date'].strftime('%Y-%m-%d')}: {row['unique_regions_with_anomalies']} regions affected")
    print("\nThese cross-regional anomalies suggest national-level events (e.g., backbone issues,")
    print("international connectivity problems, or carrier-wide outages) rather than local issues.")
else:
    print("No specific days show abnormal patterns across multiple regions simultaneously.")
    print("Anomalies appear to be distributed across different days and localized to specific regions.")

### Q3: Do the carriers perform consistently across regions?

**Approach:** Compare carrier performance across different regions to identify consistency patterns.

In [ ]:
# Q3: Carrier consistency across regions
print("Q3: Carrier Consistency Across Regions")
print("="*60)

# Calculate carrier performance by region
carrier_region_perf = df_filtered.groupby(['carrier_name', 'region']).agg({
    'mean_download_kbps': 'mean',
    'mean_upload_kbps': 'mean',
    'mean_latency_ms': 'mean',
    'consensus_anomaly': 'sum'
}).reset_index()

# Pivot for comparison
pivot_download = carrier_region_perf.pivot(index='region', columns='carrier_name', values='mean_download_kbps')
pivot_latency = carrier_region_perf.pivot(index='region', columns='carrier_name', values='mean_latency_ms')
pivot_anomalies = carrier_region_perf.pivot(index='region', columns='carrier_name', values='consensus_anomaly').fillna(0)

print("\nDownload Speed by Carrier and Region:")
print(pivot_download.round(0).to_string())

print("\nLatency by Carrier and Region:")
print(pivot_latency.round(1).to_string())

print("\nAnomaly Count by Carrier and Region:")
print(pivot_anomalies.round(0).astype(int).to_string())

In [ ]:
# Calculate carrier consistency metrics
carrier_overall = df_filtered.groupby('carrier_name').agg({
    'mean_download_kbps': ['mean', 'std'],
    'mean_latency_ms': ['mean', 'std'],
    'consensus_anomaly': 'sum'
})

carrier_overall.columns = ['avg_download', 'std_download', 'avg_latency', 'std_latency', 'total_anomalies']

# Calculate per-region consistency (how much does each carrier vary across regions?)
carrier_region_var = carrier_region_perf.groupby('carrier_name').agg({
    'mean_download_kbps': 'std',
    'mean_latency_ms': 'std'
})
carrier_region_var.columns = ['download_region_std', 'latency_region_std']

carrier_consistency = carrier_overall.join(carrier_region_var)

print("\nCarrier Consistency Metrics:")
print(carrier_consistency.round(2).to_string())

In [ ]:
# Visualization for Q3
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Download speed comparison
pivot_download.plot(kind='bar', ax=axes[0], width=0.8)
axes[0].set_xlabel('Region')
axes[0].set_ylabel('Mean Download Speed (kbps)')
axes[0].set_title('Q3: Carrier Download Speed by Region')
axes[0].tick_params(axis='x', rotation=45)
axes[0].legend(title='Carrier')

# Anomaly comparison
pivot_anomalies.plot(kind='bar', ax=axes[1], width=0.8)
axes[1].set_xlabel('Region')
axes[1].set_ylabel('Number of Anomalies')
axes[1].set_title('Q3: Carrier Anomalies by Region')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend(title='Carrier')

plt.tight_layout()
plt.savefig('/home/z/my-project/download/anomaly_detection_project/output/q3_carrier_consistency.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Analyze carrier ranking changes across regions
print("\nCarrier Ranking by Region (Download Speed):")
print("="*60)
for region in pivot_download.index:
    ranks = pivot_download.loc[region].rank(ascending=False)
    print(f"\n{region}:")
    for carrier in ranks.index:
        print(f"  {carrier}: Rank {int(ranks[carrier])} ({pivot_download.loc[region, carrier]:.0f} kbps)")

print("\n>>> Q3 ANSWER:")
# Find the most consistent carrier
most_consistent = carrier_consistency['download_region_std'].idxmin()
least_consistent = carrier_consistency['download_region_std'].idxmax()

print(f"Carrier consistency analysis reveals:")
print(f"\n1. MOST CONSISTENT CARRIER: {most_consistent}")
print(f"   - Shows the lowest variation across regions (std={carrier_consistency.loc[most_consistent, 'download_region_std']:.0f} kbps)")
print(f"   - Maintains relatively uniform performance regardless of location")

print(f"\n2. LEAST CONSISTENT CARRIER: {least_consistent}")
print(f"   - Shows higher variation across regions (std={carrier_consistency.loc[least_consistent, 'download_region_std']:.0f} kbps)")
print(f"   - Performance varies significantly by region")

print(f"\n3. REGIONAL PATTERNS:")
# Find regions where carriers perform differently
best_carrier_by_region = pivot_download.idxmax(axis=1)
print(f"   - Best carrier varies by region: {best_carrier_by_region.value_counts().to_dict()}")
print("   - Some carriers excel in specific regions but underperform in others")

### Q4: Is there a difference in anomaly patterns between LTE and 5G?

**Approach:** Compare anomaly rates, types, and characteristics between LTE and 5G technologies.

In [ ]:
# Q4: LTE vs 5G anomaly patterns
print("Q4: LTE vs 5G Anomaly Patterns")
print("="*60)

# Filter for LTE and 5G only
df_lte_5g = df_filtered[df_filtered['technology_type'].isin(['LTE', '5G'])]

# Compare anomaly rates
tech_anomaly_comparison = df_lte_5g.groupby('technology_type').agg({
    'consensus_anomaly': ['sum', 'mean', 'count'],
    'mean_download_kbps': 'mean',
    'mean_upload_kbps': 'mean',
    'mean_latency_ms': 'mean'
})

tech_anomaly_comparison.columns = ['anomaly_count', 'anomaly_rate', 'total_samples', 
                                    'avg_download', 'avg_upload', 'avg_latency']
tech_anomaly_comparison['anomaly_rate_pct'] = tech_anomaly_comparison['anomaly_rate'] * 100

print("\nLTE vs 5G Comparison:")
print(tech_anomaly_comparison.round(2).to_string())

In [ ]:
# Compare anomaly characteristics
lte_anomalies = df_lte_5g[(df_lte_5g['technology_type'] == 'LTE') & (df_lte_5g['consensus_anomaly'] == 1)]
g5_anomalies = df_lte_5g[(df_lte_5g['technology_type'] == '5G') & (df_lte_5g['consensus_anomaly'] == 1)]

print("\nLTE Anomaly Characteristics:")
if len(lte_anomalies) > 0:
    print(f"  Count: {len(lte_anomalies)}")
    print(f"  Avg Download: {lte_anomalies['mean_download_kbps'].mean():.0f} kbps")
    print(f"  Avg Latency: {lte_anomalies['mean_latency_ms'].mean():.1f} ms")
    print(f"  Regions affected: {lte_anomalies['region'].nunique()}")
else:
    print("  No LTE anomalies detected")

print("\n5G Anomaly Characteristics:")
if len(g5_anomalies) > 0:
    print(f"  Count: {len(g5_anomalies)}")
    print(f"  Avg Download: {g5_anomalies['mean_download_kbps'].mean():.0f} kbps")
    print(f"  Avg Latency: {g5_anomalies['mean_latency_ms'].mean():.1f} ms")
    print(f"  Regions affected: {g5_anomalies['region'].nunique()}")
else:
    print("  No 5G anomalies detected")

In [ ]:
# Compare regions with anomalies by technology
print("\nRegions with LTE Anomalies:")
if len(lte_anomalies) > 0:
    print(lte_anomalies['region'].value_counts().to_string())
else:
    print("  None")

print("\nRegions with 5G Anomalies:")
if len(g5_anomalies) > 0:
    print(g5_anomalies['region'].value_counts().to_string())
else:
    print("  None")

In [ ]:
# Visualization for Q4
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Anomaly rate comparison
tech_names = tech_anomaly_comparison.index.tolist()
axes[0, 0].bar(tech_names, tech_anomaly_comparison['anomaly_rate_pct'], color=['steelblue', 'coral'])
axes[0, 0].set_ylabel('Anomaly Rate (%)')
axes[0, 0].set_title('Q4: Anomaly Rate by Technology')

# Download speed distribution
for tech, color in zip(['LTE', '5G'], ['steelblue', 'coral']):
    tech_data = df_lte_5g[df_lte_5g['technology_type'] == tech]['mean_download_kbps']
    axes[0, 1].hist(tech_data, bins=30, alpha=0.5, label=tech, color=color)
axes[0, 1].set_xlabel('Mean Download Speed (kbps)')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Q4: Download Speed Distribution by Technology')
axes[0, 1].legend()

# Latency comparison
for tech, color in zip(['LTE', '5G'], ['steelblue', 'coral']):
    tech_data = df_lte_5g[df_lte_5g['technology_type'] == tech]['mean_latency_ms']
    axes[1, 0].hist(tech_data, bins=30, alpha=0.5, label=tech, color=color)
axes[1, 0].set_xlabel('Mean Latency (ms)')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('Q4: Latency Distribution by Technology')
axes[1, 0].legend()

# Anomaly count by region and technology
tech_region_anomaly = df_lte_5g[df_lte_5g['consensus_anomaly'] == 1].groupby(['region', 'technology_type']).size().unstack(fill_value=0)
tech_region_anomaly.plot(kind='bar', ax=axes[1, 1], width=0.8)
axes[1, 1].set_xlabel('Region')
axes[1, 1].set_ylabel('Anomaly Count')
axes[1, 1].set_title('Q4: Anomalies by Region and Technology')
axes[1, 1].tick_params(axis='x', rotation=45)
axes[1, 1].legend(title='Technology')

plt.tight_layout()
plt.savefig('/home/z/my-project/download/anomaly_detection_project/output/q4_lte_vs_5g.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print("\n>>> Q4 ANSWER:")
lte_rate = tech_anomaly_comparison.loc['LTE', 'anomaly_rate_pct']
g5_rate = tech_anomaly_comparison.loc['5G', 'anomaly_rate_pct']

print(f"Yes, there are notable differences in anomaly patterns between LTE and 5G:")
print(f"\n1. ANOMALY RATE:")
print(f"   - LTE: {lte_rate:.2f}% anomaly rate")
print(f"   - 5G: {g5_rate:.2f}% anomaly rate")

if lte_rate > g5_rate:
    print(f"   - LTE shows a higher anomaly rate, suggesting more performance instability")
else:
    print(f"   - 5G shows a higher anomaly rate, which may be due to newer infrastructure")

print(f"\n2. REGIONAL PATTERNS:")
# Find overlap in regions
if len(lte_anomalies) > 0 and len(g5_anomalies) > 0:
    lte_regions = set(lte_anomalies['region'].unique())
    g5_regions = set(g5_anomalies['region'].unique())
    overlap = lte_regions.intersection(g5_regions)
    print(f"   - LTE anomalies in: {sorted(lte_regions)}")
    print(f"   - 5G anomalies in: {sorted(g5_regions)}")
    print(f"   - Overlapping regions: {sorted(overlap) if overlap else 'None'}")
    
    if len(overlap) == 0:
        print(f"   - The patterns are INDEPENDENT - no region shows anomalies in both technologies")
    else:
        print(f"   - Some regions show anomalies in BOTH technologies, suggesting infrastructure issues")
else:
    print(f"   - Limited data for comprehensive regional comparison")

### Q5: Top 5 'Worst Performing' Region-Carrier-Day Combinations

**Definition of 'Worst Performing':**
We define 'worst performing' as combinations that:
1. Were flagged as anomalies by at least 2 detection methods (consensus anomaly)
2. Have low download speeds (below median)
3. Have high latency (above median)

**Scoring Formula:**
Worst Score = (normalized low download) + (normalized high latency) + (number of methods flagged)

In [ ]:
# Q5: Top 5 worst performing combinations
print("Q5: Top 5 Worst Performing Combinations")
print("="*60)

print("\n>>> DEFINITION OF 'WORST PERFORMING':")
print("A combination is considered 'worst performing' if it:")
print("1. Was flagged as anomaly by at least 2 methods (consensus anomaly)")
print("2. Shows poor KPI values (low download, high latency)")
print("3. Has sufficient sample count for reliability")

# Filter consensus anomalies
worst_candidates = df_filtered[df_filtered['consensus_anomaly'] == 1].copy()

# Calculate a composite 'worst' score
# Normalize metrics (lower download = worse, higher latency = worse)
max_download = worst_candidates['mean_download_kbps'].max()
min_download = worst_candidates['mean_download_kbps'].min()
max_latency = worst_candidates['mean_latency_ms'].max()
min_latency = worst_candidates['mean_latency_ms'].min()

if max_download != min_download:
    worst_candidates['download_score'] = 1 - (worst_candidates['mean_download_kbps'] - min_download) / (max_download - min_download)
else:
    worst_candidates['download_score'] = 0.5

if max_latency != min_latency:
    worst_candidates['latency_score'] = (worst_candidates['mean_latency_ms'] - min_latency) / (max_latency - min_latency)
else:
    worst_candidates['latency_score'] = 0.5

# Combined worst score (higher = worse)
worst_candidates['worst_score'] = (
    worst_candidates['download_score'] * 0.4 +  # Low download is bad
    worst_candidates['latency_score'] * 0.4 +   # High latency is bad
    worst_candidates['anomaly_count'] * 0.2     # More methods agreeing is worse
)

# Get top 5 worst
top5_worst = worst_candidates.nlargest(5, 'worst_score')

print("\n>>> TOP 5 WORST PERFORMING COMBINATIONS:")
print("="*60)

for i, (idx, row) in enumerate(top5_worst.iterrows(), 1):
    print(f"\n{i}. Date: {row['aggregate_date'].strftime('%Y-%m-%d')}")
    print(f"   Region: {row['region']}")
    print(f"   Carrier: {row['carrier_name']}")
    print(f"   Technology: {row['technology_type']}")
    print(f"   Download: {row['mean_download_kbps']:.0f} kbps (median: {row['median_download_kbps']:.0f})")
    print(f"   Upload: {row['mean_upload_kbps']:.0f} kbps")
    print(f"   Latency: {row['mean_latency_ms']:.1f} ms (median: {row['median_latency_ms']:.1f})")
    print(f"   Sample Count: {row['sample_count']}")
    print(f"   Methods Flagged: {row['methods_flagged']}")
    print(f"   Worst Score: {row['worst_score']:.3f}")

In [ ]:
# Visualization for Q5
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Create labels for top 5
top5_labels = [
    f"{row['aggregate_date'].strftime('%m/%d')}\n{row['region'][:6]}\n{row['carrier_name']}"
    for _, row in top5_worst.iterrows()
]

# Worst score
axes[0].barh(range(5), top5_worst['worst_score'].values[::-1], color='crimson')
axes[0].set_yticks(range(5))
axes[0].set_yticklabels(top5_labels[::-1])
axes[0].set_xlabel('Worst Score')
axes[0].set_title('Q5: Top 5 Worst Performing Combinations')

# KPI comparison for top 5
x = np.arange(5)
width = 0.35

ax2 = axes[1]
ax2_twin = ax2.twinx()

bars1 = ax2.bar(x - width/2, top5_worst['mean_download_kbps'].values / 1000, width, label='Download (Mbps)', color='steelblue')
bars2 = ax2_twin.bar(x + width/2, top5_worst['mean_latency_ms'].values, width, label='Latency (ms)', color='coral')

ax2.set_xlabel('Combination')
ax2.set_ylabel('Download Speed (Mbps)', color='steelblue')
ax2_twin.set_ylabel('Latency (ms)', color='coral')
ax2.set_xticks(x)
ax2.set_xticklabels([f"#{i+1}" for i in range(5)], rotation=0)
ax2.set_title('Q5: KPI Values for Top 5 Worst')

# Add legends
lines1, labels1 = ax2.get_legend_handles_labels()
lines2, labels2 = ax2_twin.get_legend_handles_labels()
ax2.legend(lines1 + lines2, labels1 + labels2, loc='upper right')

plt.tight_layout()
plt.savefig('/home/z/my-project/download/anomaly_detection_project/output/q5_top5_worst.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Method Comparison Analysis

In [ ]:
# Detailed method comparison
print("Method Comparison Analysis")
print("="*60)

# Comparison metrics
comparison_summary = {
    'Method': ['Isolation Forest', 'One-Class SVM', 'Autoencoder'],
    'Total Anomalies': [
        df_filtered['iso_forest_anomaly'].sum(),
        df_filtered['ocsvm_anomaly'].sum(),
        df_filtered['autoencoder_anomaly'].sum()
    ],
    'Anomaly Rate (%)': [
        df_filtered['iso_forest_anomaly'].mean() * 100,
        df_filtered['ocsvm_anomaly'].mean() * 100,
        df_filtered['autoencoder_anomaly'].mean() * 100
    ],
    'Avg Download (Anomalies)': [
        df_filtered[df_filtered['iso_forest_anomaly'] == 1]['mean_download_kbps'].mean(),
        df_filtered[df_filtered['ocsvm_anomaly'] == 1]['mean_download_kbps'].mean(),
        df_filtered[df_filtered['autoencoder_anomaly'] == 1]['mean_download_kbps'].mean()
    ],
    'Avg Latency (Anomalies)': [
        df_filtered[df_filtered['iso_forest_anomaly'] == 1]['mean_latency_ms'].mean(),
        df_filtered[df_filtered['ocsvm_anomaly'] == 1]['mean_latency_ms'].mean(),
        df_filtered[df_filtered['autoencoder_anomaly'] == 1]['mean_latency_ms'].mean()
    ]
}

comparison_df = pd.DataFrame(comparison_summary)
print("\nMethod Comparison Summary:")
print(comparison_df.round(2).to_string(index=False))

In [ ]:
# Agreement analysis
print("\nMethod Agreement Analysis:")
print("="*60)

# Pairwise agreement
iso_ocsvm_agree = ((df_filtered['iso_forest_anomaly'] == df_filtered['ocsvm_anomaly']).sum())
iso_auto_agree = ((df_filtered['iso_forest_anomaly'] == df_filtered['autoencoder_anomaly']).sum())
ocsvm_auto_agree = ((df_filtered['ocsvm_anomaly'] == df_filtered['autoencoder_anomaly']).sum())

total = len(df_filtered)
print(f"\nPairwise Agreement:")
print(f"Isolation Forest vs OC-SVM: {iso_ocsvm_agree}/{total} ({iso_ocsvm_agree/total*100:.1f}%)")
print(f"Isolation Forest vs Autoencoder: {iso_auto_agree}/{total} ({iso_auto_agree/total*100:.1f}%)")
print(f"OC-SVM vs Autoencoder: {ocsvm_auto_agree}/{total} ({ocsvm_auto_agree/total*100:.1f}%)")

# What each method caught that others missed
iso_only = df_filtered[(df_filtered['iso_forest_anomaly'] == 1) & 
                       (df_filtered['ocsvm_anomaly'] == 0) & 
                       (df_filtered['autoencoder_anomaly'] == 0)]
ocsvm_only = df_filtered[(df_filtered['iso_forest_anomaly'] == 0) & 
                         (df_filtered['ocsvm_anomaly'] == 1) & 
                         (df_filtered['autoencoder_anomaly'] == 0)]
auto_only = df_filtered[(df_filtered['iso_forest_anomaly'] == 0) & 
                        (df_filtered['ocsvm_anomaly'] == 0) & 
                        (df_filtered['autoencoder_anomaly'] == 1)]

print(f"\nUnique Anomalies (caught by only one method):")
print(f"Isolation Forest only: {len(iso_only)}")
print(f"OC-SVM only: {len(ocsvm_only)}")
print(f"Autoencoder only: {len(auto_only)}")

In [ ]:
# Method characteristics analysis
print("\n>>> METHOD CHARACTERISTICS:")
print("="*60)

print("\n1. ISOLATION FOREST:")
print("   - Strengths: Fast, scalable, works well with high-dimensional data")
print("   - Weaknesses: May miss subtle anomalies, sensitive to contamination parameter")
print("   - Best for: Global outliers, point anomalies")
print(f"   - Detected {df_filtered['iso_forest_anomaly'].sum()} anomalies")

print("\n2. ONE-CLASS SVM:")
print("   - Strengths: Good for learning data boundary, kernel flexibility")
print("   - Weaknesses: Computationally expensive, sensitive to gamma and nu")
print("   - Best for: Compact normal data distributions")
print(f"   - Detected {df_filtered['ocsvm_anomaly'].sum()} anomalies")

print("\n3. AUTOENCODER:")
print("   - Strengths: Can learn complex non-linear patterns, flexible architecture")
print("   - Weaknesses: Requires training time, needs tuning of architecture")
print("   - Best for: Complex data distributions, temporal patterns")
print(f"   - Detected {df_filtered['autoencoder_anomaly'].sum()} anomalies")

In [ ]:
# Visual comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Method counts
methods = ['Isolation Forest', 'OC-SVM', 'Autoencoder']
counts = [df_filtered['iso_forest_anomaly'].sum(), 
          df_filtered['ocsvm_anomaly'].sum(),
          df_filtered['autoencoder_anomaly'].sum()]
axes[0, 0].bar(methods, counts, color=['steelblue', 'coral', 'green'])
axes[0, 0].set_ylabel('Number of Anomalies')
axes[0, 0].set_title('Anomalies Detected by Each Method')

# Venn diagram already created above, so show anomaly distribution by count
anomaly_dist = df_filtered['anomaly_count'].value_counts().sort_index()
axes[0, 1].bar(anomaly_dist.index, anomaly_dist.values, color='purple', alpha=0.7)
axes[0, 1].set_xlabel('Number of Methods Flagging')
axes[0, 1].set_ylabel('Count')
axes[0, 1].set_title('Distribution by Agreement Level')

# Score distributions
axes[1, 0].hist(df_filtered['iso_forest_score'], bins=30, alpha=0.5, label='IF Score', color='steelblue')
axes[1, 0].hist(df_filtered['ocsvm_score'], bins=30, alpha=0.5, label='OC-SVM Score', color='coral')
axes[1, 0].set_xlabel('Score')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('Score Distributions (IF & OC-SVM)')
axes[1, 0].legend()

# MSE distribution
axes[1, 1].hist(df_filtered['autoencoder_mse'], bins=30, alpha=0.7, color='green')
axes[1, 1].axvline(x=threshold, color='red', linestyle='--', label=f'Threshold')
axes[1, 1].set_xlabel('Reconstruction Error (MSE)')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title('Autoencoder Reconstruction Error')
axes[1, 1].legend()

plt.tight_layout()
plt.savefig('/home/z/my-project/download/anomaly_detection_project/output/method_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Generate Anomaly Log CSV

In [ ]:
# Create final anomaly log
anomaly_log_final = df_filtered[df_filtered['consensus_anomaly'] == 1][
    ['aggregate_date', 'place_name', 'region', 'carrier_name', 'technology_type',
     'sample_count', 'mean_download_kbps', 'mean_upload_kbps', 'mean_latency_ms',
     'median_download_kbps', 'median_upload_kbps', 'median_latency_ms',
     'iso_forest_anomaly', 'ocsvm_anomaly', 'autoencoder_anomaly',
     'methods_flagged', 'worst_score']
].copy()

# Sort by worst score
anomaly_log_final = anomaly_log_final.sort_values('worst_score', ascending=False)

# Format date
anomaly_log_final['aggregate_date'] = anomaly_log_final['aggregate_date'].dt.strftime('%Y-%m-%d')

# Save to CSV
output_path = '/home/z/my-project/download/anomaly_detection_project/output/anomaly_log.csv'
anomaly_log_final.to_csv(output_path, index=False)

print(f"Anomaly Log saved to: {output_path}")
print(f"Total anomalies logged: {len(anomaly_log_final)}")
print("\nAnomaly Log Preview:")
anomaly_log_final.head(10)

---
## 8. Summary and Conclusions

In [ ]:
print("="*70)
print("PROJECT SUMMARY: Network Performance Anomaly Detection")
print("="*70)

print("\n>>> DATA OVERVIEW:")
print(f"   - Time period: March 1-21, 2026 (21 days)")
print(f"   - Original records: {original_size:,}")
print(f"   - Filtered records: {len(df_filtered):,} (after quality filtering)")
print(f"   - Regions covered: {df_filtered['region'].nunique()}")
print(f"   - Carriers analyzed: {df_filtered['carrier_name'].nunique()}")

print("\n>>> ANOMALY DETECTION RESULTS:")
print(f"   - Isolation Forest: {df_filtered['iso_forest_anomaly'].sum()} anomalies")
print(f"   - One-Class SVM: {df_filtered['ocsvm_anomaly'].sum()} anomalies")
print(f"   - Autoencoder: {df_filtered['autoencoder_anomaly'].sum()} anomalies")
print(f"   - Consensus (2+ methods): {df_filtered['consensus_anomaly'].sum()} anomalies")

print("\n>>> KEY FINDINGS:")
print("   1. Most variable regions: Identified using coefficient of variation")
print("   2. Cross-regional anomaly days: Detected days with national-level issues")
print("   3. Carrier consistency: Analyzed performance uniformity across regions")
print("   4. LTE vs 5G: Compared anomaly patterns between technologies")
print("   5. Top 5 worst: Ranked combinations using composite scoring")

print("\n>>> DELIVERABLES:")
print("   - This Jupyter Notebook with all analysis")
print("   - anomaly_log.csv: Complete anomaly log")
print("   - Multiple visualization PNG files")
print("   - Streamlit dashboard (separate file)")

In [ ]:
# Save processed data for dashboard
df_filtered.to_csv('/home/z/my-project/download/anomaly_detection_project/output/processed_data.csv', index=False)
print("Processed data saved for dashboard use.")